<a href="https://colab.research.google.com/github/Rudra-Sharma-432/AI_SocialMedia_Post_Automation/blob/main/social_media_studio_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Social Media Content Studio

A simple AI automation pipeline: give it a topic, it plans a post, writes it, checks it, and (optionally) generates an image for it.

In [ ]:
# Install what we need
!pip -q install -U google-genai pillow python-slugify huggingface_hub


In [ ]:
import re
import json
import time
from datetime import datetime
from pathlib import Path

from google import genai
from google.colab import userdata
from IPython.display import Markdown, display, HTML
from slugify import slugify

MODEL_NAME = "gemini-2.5-flash"
IMAGE_MODEL = "black-forest-labs/FLUX.1-schnell"

# Where we'll save everything
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Connect to Gemini
API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Add your GEMINI_API_KEY in Colab Secrets first.")

client = genai.Client(api_key=API_KEY)
print("Gemini connected. Model:", MODEL_NAME)

In [ ]:
# Redefining helper to ensure it uses the updated MODEL_NAME and SDK attributes
def ask_gemini(prompt, image=None, temperature=0.6):
    contents = [prompt] if image is None else [prompt, image]
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config={"temperature": temperature},
        )
        # Ensure we access the text property correctly for the google-genai SDK
        text = response.text.strip() if response.text else ""
        return text if text else None
    except Exception as e:
        print(f"Gemini call failed ({MODEL_NAME}): {e}")
        return None

In [ ]:
# Step 1: Get the project details from the user
print("Tell me about your campaign:\n")

topic = input("Topic: ").strip()
platform = input("Platform (LinkedIn / Instagram / X): ").strip()
audience = input("Target audience: ").strip()
goal = input("Goal (Sales / Awareness / Leads / Engagement): ").strip()

print("\nOptional, press Enter to skip:\n")
description = input("Description: ").strip()
brand_name = input("Brand name: ").strip()
image_idea = input("Basic image idea (only needed if you want an image): ").strip()

if not (topic and platform and audience and goal):
    raise ValueError("Topic, platform, audience, and goal are required.")

# This is the shared brief we'll reuse in our prompts below
brief = f"""Topic: {topic}
Platform: {platform}
Audience: {audience}
Goal: {goal}
Description: {description or "not provided"}
Brand: {brand_name or "not provided"}"""

print("\nGot it:\n")
print(brief)


In [ ]:
# Step 2: Ask Gemini if it needs more info before writing the post
gap_check = ask_gemini(f"""You are a social media strategist. Here is the brief:

{brief}

If this is enough to write a great post, reply with exactly: READY
Otherwise, ask 3-5 short, specific follow-up questions as a numbered list.""", temperature=0.2)

extra_details = ""

if gap_check and gap_check.strip().upper() != "READY":
    print(gap_check)
    print()
    for line in gap_check.splitlines():
        line = line.strip()
        if len(line) > 2 and line[0].isdigit():
            question = line.split(".", 1)[-1].strip()
            answer = input(f"{question}\n> ").strip()
            if answer:
                extra_details += f"{question} {answer}\n"
    if extra_details:
        brief += f"\nAdditional details:\n{extra_details}"

print("\nReady to generate content.")


In [ ]:
# Step 3: Generate the post
content_prompt = f"""You are a social media strategist and copywriter.

{brief}

Write a post using ONLY the information above. Do not invent facts.

Use exactly this format:

# Hook
A strong opening line.

# Main Post
The full post, optimized for {platform}.

# Hashtags
Exactly 8 relevant hashtags.

# Call To Action
One clear call to action."""

# Re-calling the helper with the updated MODEL_NAME
post = ask_gemini(content_prompt, temperature=0.7)

if not post:
    raise RuntimeError(f"Content generation failed for {MODEL_NAME}. Please check your API quota.")

display(Markdown(post))

In [ ]:
# Step 4: A couple of simple checks (no AI needed for this)
hashtag_count = len(re.findall(r"#\w+", post))

platform_limits = {"linkedin": 3000, "instagram": 2200, "x": 280, "twitter": 280}
char_limit = platform_limits.get(platform.strip().lower(), 3000)

print(f"Hashtags found: {hashtag_count} (target: 8)")
print(f"Character count: {len(post)} / {char_limit} limit for {platform}")

if hashtag_count < 6 or hashtag_count > 10 or len(post) > char_limit:
    print("\nSomething's off, asking Gemini for a quick fix...")
    fix_prompt = f"""Here is a social media post with an issue: either the hashtag count
isn't close to 8, or it's too long for {platform} (limit {char_limit} characters).

{post}

Rewrite it fixing that issue, keeping the same format:
# Hook
# Main Post
# Hashtags
# Call To Action"""
    fixed = ask_gemini(fix_prompt, temperature=0.4)
    if fixed:
        post = fixed
        display(Markdown(post))
    else:
        print("Could not fix automatically, keeping original.")
else:
    print("\nLooks good, no changes needed.")


In [ ]:
# Step 5: Turn it into a clean, copy-paste ready post (plain Python, no AI needed)
def extract_section(text, heading):
    pattern = rf"#\s*{heading}\s*\n(.*?)(?=\n#|\Z)"
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else ""

hook = extract_section(post, "Hook")
main_post = extract_section(post, "Main Post")
hashtags_text = extract_section(post, "Hashtags")
cta = extract_section(post, "Call To Action")

hashtags_clean = " ".join(re.findall(r"#\w+", hashtags_text))

publish_ready = "\n\n".join(part for part in [hook, main_post, cta, hashtags_clean] if part)

print(publish_ready)

display(HTML(f"""
<div style="background:#282a2c;color:#eee;padding:20px;border-radius:12px;
font-family:Arial;white-space:pre-wrap;font-size:16px;">
{publish_ready}
</div>
"""))


In [ ]:
# Step 6: Do you want an image too?
want_image = input("Generate an image for this post? (y/n): ").strip().lower() == "y"


In [ ]:
# Step 7: Build the image prompt and generate the image (only runs if you said yes)
generated_image = None
image_path = None

if want_image:
    image_prompt = ask_gemini(f"""You are an AI image prompt engineer.

Topic: {topic}
Platform: {platform}
Brand: {brand_name or "not provided"}
User's image idea: {image_idea or "not provided"}
Post context: {main_post}

Write ONE detailed, high-quality AI image generation prompt for this post.
Mention subject, style, lighting, colors, and mood.
Return ONLY the prompt, nothing else.""", temperature=0.6)

    print("Image prompt:\n")
    print(image_prompt)

    if image_prompt:
        from huggingface_hub import InferenceClient

        HF_TOKEN = userdata.get("HF_TOKEN")
        if not HF_TOKEN:
            print("\nHF_TOKEN not found in Colab Secrets, skipping image generation.")
        else:
            hf_client = InferenceClient(api_key=HF_TOKEN)
            try:
                generated_image = hf_client.text_to_image(prompt=image_prompt, model=IMAGE_MODEL)
                display(generated_image)

                image_path = OUTPUT_DIR / f"{TIMESTAMP}_{slugify(topic)}.png"
                generated_image.save(image_path)
                print(f"\nImage saved to {image_path}")
            except Exception as e:
                print("Image generation failed:", e)
else:
    print("Skipping image generation.")


In [ ]:
# Step 8: Save everything
campaign = {
    "timestamp": TIMESTAMP,
    "topic": topic,
    "platform": platform,
    "audience": audience,
    "goal": goal,
    "final_post": post,
    "publish_ready_post": publish_ready,
    "image_generated": want_image,
    "image_path": str(image_path) if image_path else None,
}

(OUTPUT_DIR / "campaign.json").write_text(json.dumps(campaign, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "publish_ready_post.txt").write_text(publish_ready)

print("Saved to the outputs/ folder:")
print("- campaign.json")
print("- publish_ready_post.txt")
if image_path:
    print(f"- {image_path.name}")

print("\nAll done! Your post is ready above, copy and publish it.")
